# FinOps LLM — QLoRA Fine-Tuning

**Base model:** `Qwen/Qwen3-4B-Instruct-2507`  
**Method:** 4-bit QLoRA with PEFT  
**Dataset:** 33,600 training examples, processed in four shards  
**Output:** FinOps LoRA adapter

This notebook consolidates the training workflow into a clean, reproducible sequence. Run cells from top to bottom in a GPU-enabled Kaggle or Colab runtime.

## 1. Install dependencies

Run once per fresh runtime. If the notebook asks for a runtime restart after package installation, restart and continue from the next section.

In [ ]:
%pip install -q -U transformers datasets accelerate peft bitsandbytes

## 2. Check the environment

In [ ]:
import os
import gc
import json
import time
import zipfile
import shutil

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime before running QLoRA training.")

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 3. Set paths and model

**Colab:** mount Google Drive.  
**Kaggle:** update `DATA_ZIP` and `WORK_DIR` to match the dataset location available in your notebook.

In [ ]:
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

# Update these paths for your runtime.
DATA_ZIP = "/content/drive/MyDrive/layer1_finops_output.zip"
WORK_DIR = "/content/layer1_finops_data"
DRIVE_OUTPUT = "/content/drive/MyDrive/finops_llm_training"

SHARD_DIR = os.path.join(WORK_DIR, "training_shards")
CHECKPOINT_DIR = os.path.join(DRIVE_OUTPUT, "checkpoints")
ADAPTER_DIR = os.path.join(DRIVE_OUTPUT, "final_adapter")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)

print("Base model:", MODEL_NAME)
print("Dataset archive:", DATA_ZIP)
print("Adapter output:", ADAPTER_DIR)

## 4. Mount Drive and extract the dataset

For Google Colab, mount Drive. In Kaggle, skip the Drive mount and set the paths above to the attached dataset or writable working directory.

In [ ]:
# Google Colab mounts Drive; on Kaggle, configure paths to the attached dataset.
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

if not os.path.isdir(SHARD_DIR):
    if not os.path.isfile(DATA_ZIP):
        raise FileNotFoundError(
            f"Dataset archive not found: {DATA_ZIP}. "
            "Update DATA_ZIP and WORK_DIR for this runtime."
        )
    os.makedirs(WORK_DIR, exist_ok=True)
    with zipfile.ZipFile(DATA_ZIP, "r") as archive:
        archive.extractall(WORK_DIR)

print("Runtime:", "Colab" if IN_COLAB else "Kaggle/other")
print("Shard directory exists:", os.path.isdir(SHARD_DIR))

## 5. Load and verify the four training shards

The original workflow uses four equal shards of 8,400 examples (33,600 total). Each shard is trained for one pass, continuing from the same in-memory model.

In [ ]:
shard_paths = [
    os.path.join(SHARD_DIR, f"layer1_finops_train_part_{i}.jsonl")
    for i in range(1, 5)
]

for shard_path in shard_paths:
    if not os.path.isfile(shard_path):
        raise FileNotFoundError(
            f"Missing shard: {shard_path}. "
            "Confirm the dataset was extracted and contains training_shards/."
        )

    with open(shard_path, "r", encoding="utf-8") as f:
        count = sum(1 for line in f if line.strip())

    print(f"{os.path.basename(shard_path)}: {count:,} examples")
    if count != 8400:
        raise ValueError(f"Expected 8,400 examples in {shard_path}, found {count:,}.")

print("Total training examples:", 4 * 8400)

## 6. Load the tokenizer and base model

The base model is loaded in 4-bit NF4 to reduce GPU memory use. The trained adapter is saved separately.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

print("Tokenizer and quantized base model loaded.")

## 7. Attach the LoRA configuration

In [ ]:
base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

## 8. Tokenize examples and train only on assistant responses

System and user tokens are masked with `-100`, so they do not contribute to the language-model loss. Only assistant response tokens are training targets.

In [ ]:
MAX_LENGTH = 512

def tokenize_example(example):
    messages = example["messages"]

    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    prompt_text = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=False,
        add_generation_prompt=True,
    )

    full_tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=False,
    )
    prompt_tokens = tokenizer(
        prompt_text,
        truncation=True,
        max_length=MAX_LENGTH,
        add_special_tokens=False,
    )

    input_ids = full_tokens["input_ids"]
    prompt_length = min(len(prompt_tokens["input_ids"]), len(input_ids))
    labels = [-100] * prompt_length + input_ids[prompt_length:]

    return {
        "input_ids": input_ids,
        "attention_mask": full_tokens["attention_mask"],
        "labels": labels,
    }

In [ ]:
from dataclasses import dataclass
from typing import Any

@dataclass
class FinOpsDataCollator:
    tokenizer: Any

    def __call__(self, features):
        labels = [feature["labels"] for feature in features]
        inputs = [
            {
                "input_ids": feature["input_ids"],
                "attention_mask": feature["attention_mask"],
            }
            for feature in features
        ]

        batch = self.tokenizer.pad(
            inputs,
            padding=True,
            return_tensors="pt",
        )

        max_length = batch["input_ids"].shape[1]
        padded_labels = [
            label + [-100] * (max_length - len(label))
            for label in labels
        ]
        batch["labels"] = torch.tensor(padded_labels, dtype=torch.long)
        return batch

data_collator = FinOpsDataCollator(tokenizer)

## 9. Train the four shards sequentially

This cell runs the full training sequence. It saves checkpoints by shard and updates the final adapter after each shard. Training can take a long time; keep the runtime active and verify available storage first.

In [ ]:
training_results = []

for part_number, shard_path in enumerate(shard_paths, start=1):
    print(f"\n===== Training shard {part_number}/4 =====")

    raw_part = load_dataset(
        "json",
        data_files={"train": shard_path},
    )["train"]

    tokenized_part = raw_part.map(
        tokenize_example,
        remove_columns=raw_part.column_names,
        desc=f"Tokenizing shard {part_number}",
    )

    part_output = os.path.join(CHECKPOINT_DIR, f"part_{part_number}")

    training_args = TrainingArguments(
        output_dir=part_output,
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=1e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        optim="paged_adamw_8bit",
        fp16=True,
        gradient_checkpointing=True,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        logging_steps=25,
        report_to="none",
        max_grad_norm=1.0,
        seed=42,
        remove_unused_columns=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_part,
        data_collator=data_collator,
    )

    start = time.time()
    result = trainer.train()
    elapsed = time.time() - start

    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)

    training_results.append({
        "part": part_number,
        "examples": len(tokenized_part),
        "seconds": round(elapsed, 2),
        "train_loss": result.training_loss,
    })

    print(f"Shard {part_number} complete in {elapsed / 60:.1f} minutes.")
    print(f"Adapter saved to: {ADAPTER_DIR}")

    del trainer, raw_part, tokenized_part
    gc.collect()
    torch.cuda.empty_cache()

print("\nAll four shards completed.")
print(training_results)

## 10. Verify and save the final adapter

The adapter folder contains the LoRA weights and configuration; it is not a standalone copy of Qwen3-4B.

In [ ]:
required_files = ["adapter_config.json", "adapter_model.safetensors"]
for filename in required_files:
    file_path = os.path.join(ADAPTER_DIR, filename)
    if not os.path.isfile(file_path):
        raise FileNotFoundError(f"Expected output file not found: {file_path}")

print("Final adapter files:")
for filename in sorted(os.listdir(ADAPTER_DIR)):
    full_path = os.path.join(ADAPTER_DIR, filename)
    if os.path.isfile(full_path):
        print(f"- {filename}: {os.path.getsize(full_path) / (1024**2):.2f} MB")

## 11. Quick inference check

This tests the trained model in the current session. Use the project's full approved FinOps system prompt when conducting formal evaluation.

In [ ]:
question = (
    "A cloud VM costs $800 per month. CPU utilization is 12% and "
    "memory utilization is 25%. The proposed 4-vCPU configuration "
    "has no known price. What can be concluded, and what evidence is missing?"
)

messages = [
    {
        "role": "system",
        "content": (
            "You are a FinOps and cloud cost optimization assistant. "
            "Use only supplied evidence. Do not invent prices or savings. "
            "Identify missing evidence."
        ),
    },
    {"role": "user", "content": question},
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
    )

new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))

## Notes and limitations

- Run in a GPU-enabled runtime; this notebook is not intended for CPU-only training.
- Paths are configured for Google Colab and may need adjustment in Kaggle.
- Training the four shards sequentially assumes one continuous runtime and the same in-memory model. If the runtime disconnects, resume from a saved adapter/checkpoint using a deliberate resume workflow rather than blindly rerunning all cells.
- The adapter depends on the compatible Qwen3-4B base model.
- Training and inference outputs should be evaluated before making claims about FinOps reasoning accuracy.
- This notebook demonstrates model training and inference; it does not deploy a persistent public API.